# Food-bank requirement prediction

This notebook trains a small PyTorch multilayer perceptron to predict `quantity_required` from food-bank, category, season, and distribution information.

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
FEATURE_COLUMNS = ["food_bank_id", "category", "season", "quantity_distributed"]
TARGET_COLUMN = "quantity_required"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
data_path = Path("data/inventory_history.csv")
if not data_path.exists():
    data_path = Path("../data/inventory_history.csv")
frame = pd.read_csv(data_path)

required_columns = set(FEATURE_COLUMNS + [TARGET_COLUMN])
missing_columns = required_columns.difference(frame.columns)
if missing_columns:
    raise ValueError(f"Missing required CSV columns: {sorted(missing_columns)}")

features = frame[FEATURE_COLUMNS].copy()
features["food_bank_id"] = features["food_bank_id"].astype(str)
categorical_columns = ["food_bank_id", "category", "season"]
category_values = {
    column: sorted(features[column].astype(str).unique().tolist())
    for column in categorical_columns
}
features = pd.get_dummies(
    features,
    columns=categorical_columns,
    dtype=np.float32,
)

targets = frame[TARGET_COLUMN].to_numpy(dtype=np.float32)
split_generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(features), generator=split_generator).numpy()
validation_size = max(1, int(len(features) * 0.2))
validation_indices = permutation[:validation_size]
training_indices = permutation[validation_size:]

distributed = features["quantity_distributed"]
training_mean = float(distributed.iloc[training_indices].mean())
training_std = float(distributed.iloc[training_indices].std()) or 1.0
features["quantity_distributed"] = (distributed - training_mean) / training_std

feature_tensor = torch.tensor(features.to_numpy(dtype=np.float32))
target_tensor = torch.tensor(targets)
training_dataset = TensorDataset(
    feature_tensor[training_indices], target_tensor[training_indices]
)
validation_dataset = TensorDataset(
    feature_tensor[validation_indices], target_tensor[validation_indices]
)
training_loader = DataLoader(training_dataset, batch_size=64, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=64)

print(f"Training records: {len(training_dataset)}")
print(f"Validation records: {len(validation_dataset)}")
print(f"Number of input features: {feature_tensor.shape[1]}")

In [ ]:
class RequirementPredictor(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, inputs):
        return self.network(inputs).squeeze(1)


model = RequirementPredictor(feature_tensor.shape[1])
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 100
accuracy_tolerance = 10.0
training_losses = []
validation_losses = []
training_accuracies = []
validation_accuracies = []

for epoch in range(1, epochs + 1):
    model.train()
    training_loss = 0.0
    training_correct = 0
    training_count = 0
    for batch_features, batch_targets in training_loader:
        optimizer.zero_grad()
        predictions = model(batch_features)
        loss = loss_function(predictions, batch_targets)
        loss.backward()
        optimizer.step()
        training_loss += loss.item() * len(batch_targets)
        training_correct += (torch.abs(predictions - batch_targets) <= accuracy_tolerance).sum().item()
        training_count += len(batch_targets)

    training_loss /= training_count
    training_accuracy = 100 * training_correct / training_count

    model.eval()
    validation_loss = 0.0
    validation_correct = 0
    validation_count = 0
    with torch.no_grad():
        for batch_features, batch_targets in validation_loader:
            predictions = model(batch_features)
            batch_loss = loss_function(predictions, batch_targets)
            validation_loss += batch_loss.item() * len(batch_targets)
            validation_correct += (torch.abs(predictions - batch_targets) <= accuracy_tolerance).sum().item()
            validation_count += len(batch_targets)
    validation_loss /= validation_count
    validation_accuracy = 100 * validation_correct / validation_count
    training_losses.append(training_loss)
    validation_losses.append(validation_loss)
    training_accuracies.append(training_accuracy)
    validation_accuracies.append(validation_accuracy)

    if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
        print(
            f"Epoch {epoch:>3}/{epochs} - "
            f"training loss: {training_loss:.4f}, validation loss: {validation_loss:.4f}, "
            f"training accuracy: {training_accuracy:.2f}%, validation accuracy: {validation_accuracy:.2f}%"
        )

In [ ]:
model.eval()
absolute_error = 0.0
sample_count = 0
with torch.no_grad():
    for batch_features, batch_targets in validation_loader:
        absolute_error += torch.abs(model(batch_features) - batch_targets).sum().item()
        sample_count += len(batch_targets)

validation_mae = absolute_error / sample_count
final_validation_accuracy = validation_accuracies[-1]
print(f"Validation MSE: {validation_loss:.4f}")
print(f"Validation MAE: {validation_mae:.4f}")
print(f"Validation accuracy within +/-{accuracy_tolerance:g} units: {final_validation_accuracy:.2f}%")

epochs_axis = range(1, epochs + 1)
figure, (loss_axis, accuracy_axis) = plt.subplots(1, 2, figsize=(14, 5))
loss_axis.plot(epochs_axis, training_losses, label="Training loss")
loss_axis.plot(epochs_axis, validation_losses, label="Validation loss")
loss_axis.set_title("Loss during training")
loss_axis.set_xlabel("Epoch")
loss_axis.set_ylabel("MSE loss")
loss_axis.legend()
loss_axis.grid(alpha=0.3)

accuracy_axis.plot(epochs_axis, training_accuracies, label="Training accuracy")
accuracy_axis.plot(epochs_axis, validation_accuracies, label="Validation accuracy")
accuracy_axis.set_title(f"Accuracy within +/-{accuracy_tolerance:g} units")
accuracy_axis.set_xlabel("Epoch")
accuracy_axis.set_ylabel("Accuracy (%)")
accuracy_axis.set_ylim(0, 100)
accuracy_axis.legend()
accuracy_axis.grid(alpha=0.3)
figure.tight_layout()
plt.show()

checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_input_size": feature_tensor.shape[1],
    "feature_names": features.columns.tolist(),
    "category_values": category_values,
    "quantity_distributed_mean": training_mean,
    "quantity_distributed_std": training_std,
    "validation_mse": validation_loss,
    "validation_mae": validation_mae,
    "validation_accuracy": final_validation_accuracy,
    "accuracy_tolerance": accuracy_tolerance,
}
output_path = Path("quantity_required_model.pt")
if not Path("data").exists():
    output_path = Path("../quantity_required_model.pt")
torch.save(checkpoint, output_path)
print(f"Saved model checkpoint to {output_path}")